In [1]:
#Author: Hiba Ali 
#E-mail: hhibaa802@gmail.com 
#Purpose: QSS Thesis

In [1]:
#import libraries 
import pandas as pd
import os

In [2]:
#read data
df = pd.read_csv("../data/politicians_list.csv") 
df.head()

,FirstName,LastName,X Handle,St/Dis,Party,Role,Past or current?
0,Alma,Adams,@RepAdams,NC12,D,Representative,Current
1,Robert,Aderholt,@Robert_Aderholt,AL04,R,Representative,Current
2,Pete,Aguilar,@RepPeteAguilar,CA33,D,Representative,Current
3,Mark,Alford,@RepMarkAlford,MO04,R,Representative,Current
4,Rick,Allen,@RepRickAllen,GA12,R,Representative,Current


In [3]:
#check shape
df.shape

#735 users total

(735, 7)

In [4]:
#check for duplicates
duplicates = df.duplicated(subset=['FirstName', 'LastName'], keep=False)

#show rows that are duplicate
df[duplicates]

,FirstName,LastName,X Handle,St/Dis,Party,Role,Past or current?
74,Lou,Correa,@RepLouCorrea,CA46,D,Representative,Current
159,Morgan,Griffith,@RepMGriffith,VA09,R,Representative,Current
438,Jim,Banks,@SenatorBanks,IN,R,Senate,Current
499,Markwayne,Mullin,@SenMullin,OK,R,Senate,Current
521,Elissa,Slotkin,@SenatorSlotkin,MI,D,Senate,Current
644,Elissa,Slotkin,@SenatorSlotkin,NaN,D,Representative,Past
654,Morgan,Griffith,@RepMGriffith,NaN,R,Representative,Past
656,Lou,Correa,@RepLouCorrea,NaN,D,Representative,Past
669,Jim,Banks,@Jim_Banks,NaN,R,Representative,Past
695,Markwayne,Mullin,@MarkwayneMullin,NaN,R,Representative,Past


In [5]:
df = df.dropna(subset=['X Handle'])
df.shape #now have 723 politicians 

(723, 7)

In [11]:
#drop duplicates

# Drop rows with specific indices
rows_to_drop = [644, 654, 656, 669, 695]
df = df.drop(index=rows_to_drop)

# Optional: reset the index
df = df.reset_index(drop=True)

df['X Handle'] = df['X Handle'].str.replace('@', '', regex=False)

print(df.shape)
#now have 718 politicians)

(718, 7)


In [1045]:
df.to_csv("df.csv", index=False)

In [13]:
# Separate the groups
sen_rep = df[df['Role'].isin(['Senator', 'Representative'])]
other = df[~df['Role'].isin(['Senator', 'Representative'])]

# Randomly sample 202 Senators/Representatives out of the 670 total, about 30%
sen_rep_sample = sen_rep.sample(n=202, random_state=42)

# Combine with all other roles
final_df = pd.concat([sen_rep_sample, other], ignore_index=True)

print(final_df.shape)  # Should be 350

(350, 7)


In [15]:
# Remove '@' from 'X Handle' column
final_df['X Handle'] = final_df['X Handle'].str.replace('@', '', regex=False)

In [ ]:
PULLING HEALTH-RELATED TWEETS AND SAVING THEM

In [23]:
import requests
import time
import json

API_BASE_URL = "https://api.data365.co"
API_VERSION = "/v1.1"
API_AUTHORIZATION_TOKEN = "ZXlKMGVYQWlPaUpLVjFRaUxDSmhiR2NpT2lKSVV6STFOaUo5LmV5SnpkV0lpT2lKSWFXSmhRV3hwSWl3aWFXRjBJam94TnpZNU5UQTNOamcyTGpBd05UQXhPSDAuNmpTLTZtbVBGV0F1ZS1McHhXaUIwbUtDUWFDemU4VkRNZVhOSkZ2ckJRTQ=="

In [25]:
# Map shortcut numbers to keyword blocks
KEYWORD_BLOCKS = {
    "1A": "health OR medicine OR medical OR medication",
    "1B": "treatment OR symptoms OR diagnosis",
    "2A": "healthcare OR insurance OR medicare OR medicaid",
    "2B": "premiums OR coverage OR patients OR hospitals",
    "3": "flu OR pneumonia OR bronchitis OR strep throat OR sore throat OR cough OR fever OR infection OR contagious",
    "4": "high blood pressure OR hypertension OR heart OR heart disease OR heart rate OR stroke OR cholesterol OR diabetes OR cancer",
    "5": "lupus OR autoimmune OR chronic illness OR inflammation OR immune system",
    "6": "hydration OR diet OR nutrition OR exercise OR workout OR sleep OR wellness"
}


In [1015]:
import time
import requests

shortcut = "1A"  
keywords = KEYWORD_BLOCKS[shortcut]

update_url = f"{API_BASE_URL}{API_VERSION}/twitter/search/post/update"
posts_url = f"{API_BASE_URL}{API_VERSION}/twitter/search/post/posts"

params = {
    "from_profile": "Liz_Cheney",
    "from_date": "2022-01-01",
    "to_date": "2025-12-31",
    "lang": "en",
    "max_posts": 20,
    "search_type": "latest",
    "max_replies": 3,
    "load_replies": "true",
    "keywords": keywords,
    "access_token": API_AUTHORIZATION_TOKEN
}

# Start the search task
post_response = requests.post(update_url, params=params)
print("Task started:", post_response.status_code)

# Poll for results
max_attempts = 10
attempt = 0
data = {}

while attempt < max_attempts:
    posts_response = requests.get(posts_url, params=params)
    if posts_response.status_code == 200:
        data = posts_response.json()
        items = data.get("data", {}).get("items", [])
        if items:
            print(f"Number of items retrieved: {len(items)}")
            break
        else:
            print(f"Attempt {attempt+1}: No tweets yet, waiting...")
    else:
        print(f"Failed to fetch posts (attempt {attempt+1}):", posts_response.status_code)
    
    attempt += 1
    time.sleep(2)  # wait 2 seconds before next attempt

if not data.get("data", {}).get("items"):
    print("No tweets were retrieved after polling.")

Task started: 202
Attempt 1: No tweets yet, waiting...
Attempt 2: No tweets yet, waiting...
Attempt 3: No tweets yet, waiting...
Attempt 4: No tweets yet, waiting...
Attempt 5: No tweets yet, waiting...
Attempt 6: No tweets yet, waiting...
Attempt 7: No tweets yet, waiting...
Attempt 8: No tweets yet, waiting...
Attempt 9: No tweets yet, waiting...
Attempt 10: No tweets yet, waiting...
No tweets were retrieved after polling.


In [915]:
print(json.dumps(data, indent=4))

{
    "data": {
        "items": [],
        "page_info": {
            "cursor": null,
            "has_next_page": false
        }
    },
    "error": null,
    "status": "ok"
}


In [983]:
politician_name = "RepJoeCourtney"   
block_name = shortcut          

filename = f"{politician_name} Block {block_name}.json"

with open(filename, "w", encoding="utf-8") as f:
    json.dump(data, f, indent=4)

print(f"Saved results to: {filename}")

Saved results to: RepJoeCourtney Block 6.json


In [ ]:
PULLING HEALTH-RELATED TWEET REPLIES AND SAVING THEM

In [985]:
username = "RepJoeCourtney"
input_filename = "RepJoeCourtney Block 6.json"
output_filename = "RepJoeCourtney Block 6 Replies.json"

# ------------------------------------
# 1. Open the saved block file
# ------------------------------------
with open(input_filename, "r", encoding="utf-8") as f:
    block_data = json.load(f)

posts = block_data.get("data", {}).get("items", [])

all_replies_output = []

# ------------------------------------
# 2. Extract every conversation_id
# ------------------------------------
for post in posts:
    conversation_id = post.get("conversation_id")

    if not conversation_id:
        continue

    print(f"Fetching replies for post {conversation_id}")

    # ------------------------------------
    # 3. Call replies endpoint
    # ------------------------------------
    replies_url = f"{API_BASE_URL}{API_VERSION}/twitter/profile/{username}/feed/posts/{conversation_id}/replies"

    params = {
        "order_by": "date_desc",
        "max_page_size": 50,
        "access_token": API_AUTHORIZATION_TOKEN
    }

    response = requests.get(replies_url, params=params)

    if response.status_code == 200:
        replies_json = response.json()
        replies_items = replies_json.get("data", {}).get("items", [])

        all_replies_output.append({
            "conversation_id": conversation_id,
            "replies": replies_items
        })

    else:
        print(f"Failed to fetch replies for {conversation_id}: {response.status_code}")

    time.sleep(1)  # small pause to avoid rate limits

# ------------------------------------
# 4. Save everything into new JSON file
# ------------------------------------
final_output = {
    "politician": username,
    "block": "6",
    "posts": all_replies_output
}

with open(output_filename, "w", encoding="utf-8") as f:
    json.dump(final_output, f, indent=4)

print("Replies file successfully created.")

Fetching replies for post 1489708580613804052
Fetching replies for post 1534524266871398401
Fetching replies for post 1573385180416716800
Fetching replies for post 1798126941448581254
Fetching replies for post 1958677352549892527
Replies file successfully created.


In [ ]:
AUTOMATING HEALTH-RELATED POSTS

In [1013]:
import pandas as pd
import requests
import time
import json

API_BASE_URL = "https://api.data365.co"
API_VERSION = "/v1.1"
API_AUTHORIZATION_TOKEN = "ZXlKMGVYQWlPaUpLVjFRaUxDSmhiR2NpT2lKSVV6STFOaUo5LmV5SnpkV0lpT2lKSWFXSmhRV3hwSWl3aWFXRjBJam94TnpZNU5UQTNOamcyTGpBd05UQXhPSDAuNmpTLTZtbVBGV0F1ZS1McHhXaUIwbUtDUWFDemU4VkRNZVhOSkZ2ckJRTQ=="

# Keyword blocks
KEYWORD_BLOCKS = {
    "1A": "health OR medicine OR medical OR medication",
    "1B": "treatment OR symptoms OR diagnosis",
    "2A": "healthcare OR insurance OR medicare OR medicaid",
    "2B": "premiums OR coverage OR patients OR hospitals",
    "3": "flu OR pneumonia OR bronchitis OR strep throat OR sore throat OR cough OR fever OR infection OR contagious",
    "4": "high blood pressure OR hypertension OR heart OR heart disease OR heart rate OR stroke OR cholesterol OR diabetes OR cancer",
    "5": "lupus OR autoimmune OR chronic illness OR inflammation OR immune system",
    "6": "hydration OR diet OR nutrition OR exercise OR workout OR sleep OR wellness"
}

# ------------------------
# Function to fetch posts only
# ------------------------
def fetch_posts(username, block_name, keywords, max_attempts=10, sleep_time=2):
    update_url = f"{API_BASE_URL}{API_VERSION}/twitter/search/post/update"
    posts_url = f"{API_BASE_URL}{API_VERSION}/twitter/search/post/posts"

    params = {
        "from_profile": username,
        "from_date": "2022-01-01",
        "to_date": "2025-12-31",
        "lang": "en",
        "max_posts": 20,
        "search_type": "latest",
        "max_replies": 3,
        "load_replies": "true",
        "keywords": keywords,
        "access_token": API_AUTHORIZATION_TOKEN
    }

    # Start the search task
    post_response = requests.post(update_url, params=params)
    print(f"Task started for {username} Block {block_name}: {post_response.status_code}")

    attempt = 0
    data = {}
    while attempt < max_attempts:
        posts_response = requests.get(posts_url, params=params)
        if posts_response.status_code == 200:
            data = posts_response.json()
            items = data.get("data", {}).get("items", [])
            if items:
                print(f"{username} Block {block_name}: Retrieved {len(items)} posts")
                break
            else:
                print(f"{username} Block {block_name}, attempt {attempt+1}: No posts yet...")
        else:
            print(f"Failed to fetch posts for {username} (attempt {attempt+1}): {posts_response.status_code}")
        attempt += 1
        time.sleep(sleep_time)

    items = data.get("data", {}).get("items", [])
    if items:
        filename = f"{username} Block {block_name}.json"
        with open(filename, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=4)
        print(f"Saved posts to {filename}")
    else:
        print(f"No posts found for {username} Block {block_name}, skipping save.")

    return items

# ------------------------
# Main Execution
# ------------------------
# Assuming final_df is your DataFrame with a 'X Handle' column
start_username = "SecDebHaaland"
start_index = final_df.index[final_df['X Handle'] == start_username][0]

for username in final_df['X Handle'].iloc[start_index:]:
    print(f"\n=== Processing {username} ===")
    for block_name, keywords in KEYWORD_BLOCKS.items():
        fetch_posts(username, block_name, keywords)


=== Processing SecDebHaaland ===
Task started for SecDebHaaland Block 1A: 202
SecDebHaaland Block 1A: Retrieved 20 posts
Saved posts to SecDebHaaland Block 1A.json
Task started for SecDebHaaland Block 1B: 202
SecDebHaaland Block 1B, attempt 1: No posts yet...
SecDebHaaland Block 1B, attempt 2: No posts yet...
SecDebHaaland Block 1B, attempt 3: No posts yet...
SecDebHaaland Block 1B, attempt 4: No posts yet...
SecDebHaaland Block 1B, attempt 5: No posts yet...
SecDebHaaland Block 1B: Retrieved 2 posts
Saved posts to SecDebHaaland Block 1B.json
Task started for SecDebHaaland Block 2A: 202
SecDebHaaland Block 2A, attempt 1: No posts yet...
SecDebHaaland Block 2A, attempt 2: No posts yet...
SecDebHaaland Block 2A, attempt 3: No posts yet...
SecDebHaaland Block 2A, attempt 4: No posts yet...
SecDebHaaland Block 2A, attempt 5: No posts yet...
SecDebHaaland Block 2A: Retrieved 4 posts
Saved posts to SecDebHaaland Block 2A.json
Task started for SecDebHaaland Block 2B: 202
SecDebHaaland Block 

In [ ]:
AUTOMATING HEALTH-RELATED REPLIES

In [1019]:
import os
import json
import requests
import time

# ------------------------------------
# CONFIG
# ------------------------------------
BLOCKS = ["1A", "1B", "2A", "2B", "3", "4", "5", "6"]  # correct block names

# ------------------------------------
# START AT MayraFlores4TX AND GO DOWNWARD
# ------------------------------------
handles_series = final_df["X Handle"].dropna()

if "MayraFlores4TX" in handles_series.values:
    start_index = handles_series[handles_series == "MayraFlores4TX"].index[0]
    usernames_to_process = final_df.loc[start_index:, "X Handle"].dropna().unique()
else:
    print("MayraFlores4TX not found. Starting from top.")
    usernames_to_process = handles_series.unique()

# ------------------------------------
# LOOP THROUGH EACH POLITICIAN
# ------------------------------------
for username in usernames_to_process:

    print(f"\nProcessing politician: {username}")

    # ------------------------------------
    # LOOP THROUGH BLOCKS
    # ------------------------------------
    for block_name in BLOCKS:

        input_filename = f"{username} Block {block_name}.json"
        output_filename = f"{username} Block {block_name} Replies.json"

        # ------------------------------------
        # SKIP IF FILE DOESN'T EXIST
        # ------------------------------------
        if not os.path.exists(input_filename):
            print(f"Skipping {input_filename} (file not found)")
            continue

        # ------------------------------------
        # OPEN BLOCK FILE
        # ------------------------------------
        with open(input_filename, "r", encoding="utf-8") as f:
            block_data = json.load(f)

        posts = block_data.get("data", {}).get("items", [])

        # Skip if no posts
        if not posts:
            print(f"Skipping {input_filename} (no posts found)")
            continue

        all_replies_output = []

        # ------------------------------------
        # FETCH REPLIES FOR EACH POST
        # ------------------------------------
        for post in posts:
            conversation_id = post.get("conversation_id")

            if not conversation_id:
                continue

            print(f"Fetching replies for {username} Block {block_name} - {conversation_id}")

            replies_url = (
                f"{API_BASE_URL}{API_VERSION}/twitter/profile/"
                f"{username}/feed/posts/{conversation_id}/replies"
            )

            params = {
                "order_by": "date_desc",
                "max_page_size": 50,
                "access_token": API_AUTHORIZATION_TOKEN
            }

            response = requests.get(replies_url, params=params)

            if response.status_code == 200:
                replies_json = response.json()
                replies_items = replies_json.get("data", {}).get("items", [])

                all_replies_output.append({
                    "conversation_id": conversation_id,
                    "replies": replies_items
                })
            else:
                print(f"Failed for {conversation_id}: {response.status_code}")

            time.sleep(1)  # rate limit safety

        # ------------------------------------
        # SAVE OUTPUT ONLY IF REPLIES EXIST
        # ------------------------------------
        if all_replies_output:

            final_output = {
                "politician": username,
                "block": block_name,
                "posts": all_replies_output
            }

            with open(output_filename, "w", encoding="utf-8") as f:
                json.dump(final_output, f, indent=4)

            print(f"Saved: {output_filename}")
        else:
            print(f"No replies found for {username} Block {block_name}")

print("\nAll politicians processed.")


Processing politician: MayraFlores4TX
Fetching replies for MayraFlores4TX Block 1A - 1508451806187302924
Fetching replies for MayraFlores4TX Block 1A - 1520137165434404865
Fetching replies for MayraFlores4TX Block 1A - 1540478880309460993
Fetching replies for MayraFlores4TX Block 1A - 1557397296484175874
Fetching replies for MayraFlores4TX Block 1A - 1558984034377760776
Fetching replies for MayraFlores4TX Block 1A - 1575972354936950784
Fetching replies for MayraFlores4TX Block 1A - 1576271103781376006
Fetching replies for MayraFlores4TX Block 1A - 1643790685277921283
Fetching replies for MayraFlores4TX Block 1A - 1646993385507500034
Fetching replies for MayraFlores4TX Block 1A - 1661018805269852166
Fetching replies for MayraFlores4TX Block 1A - 1694432554412048692
Fetching replies for MayraFlores4TX Block 1A - 1770471980510101606
Fetching replies for MayraFlores4TX Block 1A - 1792558727272313065
Fetching replies for MayraFlores4TX Block 1A - 1807053284403802280
Fetching replies for Ma

In [ ]:
GET NON-HEALTH TWEETS FOR FIRST HALF

In [1023]:
import pandas as pd
import requests
import time
import json

API_BASE_URL = "https://api.data365.co"
API_VERSION = "/v1.1"
API_AUTHORIZATION_TOKEN = "ZXlKMGVYQWlPaUpLVjFRaUxDSmhiR2NpT2lKSVV6STFOaUo5LmV5SnpkV0lpT2lKSWFXSmhRV3hwSWl3aWFXRjBJam94TnpZNU5UQTNOamcyTGpBd05UQXhPSDAuNmpTLTZtbVBGV0F1ZS1McHhXaUIwbUtDUWFDemU4VkRNZVhOSkZ2ckJRTQ=="

# ------------------------
# Function to fetch top posts (no keywords)
# ------------------------
def fetch_top_posts(username, max_attempts=10, sleep_time=2):

    update_url = f"{API_BASE_URL}{API_VERSION}/twitter/search/post/update"
    posts_url = f"{API_BASE_URL}{API_VERSION}/twitter/search/post/posts"

    params = {
        "from_profile": username,
        "from_date": "2022-01-01",
        "to_date": "2025-12-31",
        "lang": "en",
        "max_posts": 20,
        "search_type": "top",   # change to "top" if API supports it
        "max_replies": 3,
        "load_replies": "true",
        "access_token": API_AUTHORIZATION_TOKEN
    }

    # Start search task
    post_response = requests.post(update_url, params=params)
    print(f"Task started for {username}: {post_response.status_code}")

    attempt = 0
    data = {}

    while attempt < max_attempts:
        posts_response = requests.get(posts_url, params=params)

        if posts_response.status_code == 200:
            data = posts_response.json()
            items = data.get("data", {}).get("items", [])

            if items:
                print(f"{username}: Retrieved {len(items)} posts")
                break
            else:
                print(f"{username}, attempt {attempt+1}: No posts yet...")
        else:
            print(f"Failed for {username} (attempt {attempt+1}): {posts_response.status_code}")

        attempt += 1
        time.sleep(sleep_time)

    items = data.get("data", {}).get("items", [])

    if items:
        filename = f"{username}.json"
        with open(filename, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=4)
        print(f"Saved posts to {filename}")
    else:
        print(f"No posts found for {username}, skipping save.")

    return items


# ------------------------
# MAIN EXECUTION
# ------------------------

start_username = "RepWexton"

# Find starting index
start_index = final_df.index[final_df["X Handle"] == start_username][0]

# Loop downward from RepWexton
for username in final_df["X Handle"].iloc[start_index:]:
    print(f"\n=== Processing {username} ===")
    fetch_top_posts(username)

print("\nAll politicians processed.")


=== Processing RepWexton ===
Task started for RepWexton: 202
RepWexton: Retrieved 25 posts
Saved posts to RepWexton.json

=== Processing TomColeOK04 ===
Task started for TomColeOK04: 202
TomColeOK04: Retrieved 25 posts
Saved posts to TomColeOK04.json

=== Processing RepLoisFrankel ===
Task started for RepLoisFrankel: 202
RepLoisFrankel: Retrieved 25 posts
Saved posts to RepLoisFrankel.json

=== Processing RepJillTokuda ===
Task started for RepJillTokuda: 202
RepJillTokuda: Retrieved 25 posts
Saved posts to RepJillTokuda.json

=== Processing michaelgwaltz ===
Task started for michaelgwaltz: 202
michaelgwaltz: Retrieved 25 posts
Saved posts to michaelgwaltz.json

=== Processing RepTerriSewell ===
Task started for RepTerriSewell: 202
RepTerriSewell: Retrieved 25 posts
Saved posts to RepTerriSewell.json

=== Processing RepJeffHurd ===
Task started for RepJeffHurd: 202
RepJeffHurd: Retrieved 25 posts
Saved posts to RepJeffHurd.json

=== Processing RepSheriBiggs ===
Task started for RepSher

In [ ]:
AUTOMATING REPLIES FOR NON-HEALTH TWEETS

In [1025]:
import os
import json
import requests
import time

# ------------------------------------
# START AT RepWexton AND GO DOWNWARD
# ------------------------------------
handles_series = final_df["X Handle"].dropna()

if "RepWexton" in handles_series.values:
    start_index = handles_series[handles_series == "RepWexton"].index[0]
    usernames_to_process = final_df.loc[start_index:, "X Handle"].dropna().unique()
else:
    print("RepWexton not found. Starting from top.")
    usernames_to_process = handles_series.unique()

# ------------------------------------
# LOOP THROUGH EACH POLITICIAN
# ------------------------------------
for username in usernames_to_process:

    print(f"\nProcessing politician: {username}")

    input_filename = f"{username}.json"
    output_filename = f"{username} Replies.json"

    # ------------------------------------
    # SKIP IF POSTS FILE DOESN'T EXIST
    # ------------------------------------
    if not os.path.exists(input_filename):
        print(f"Skipping {input_filename} (file not found)")
        continue

    # ------------------------------------
    # OPEN POSTS FILE
    # ------------------------------------
    with open(input_filename, "r", encoding="utf-8") as f:
        post_data = json.load(f)

    posts = post_data.get("data", {}).get("items", [])

    if not posts:
        print(f"No posts found in {input_filename}, skipping.")
        continue

    all_replies_output = []

    # ------------------------------------
    # FETCH REPLIES FOR EACH POST
    # ------------------------------------
    for post in posts:
        conversation_id = post.get("conversation_id")

        if not conversation_id:
            continue

        print(f"Fetching replies for {username} - {conversation_id}")

        replies_url = (
            f"{API_BASE_URL}{API_VERSION}/twitter/profile/"
            f"{username}/feed/posts/{conversation_id}/replies"
        )

        params = {
            "order_by": "date_desc",
            "max_page_size": 50,
            "access_token": API_AUTHORIZATION_TOKEN
        }

        response = requests.get(replies_url, params=params)

        if response.status_code == 200:
            replies_json = response.json()
            replies_items = replies_json.get("data", {}).get("items", [])

            all_replies_output.append({
                "conversation_id": conversation_id,
                "replies": replies_items
            })
        else:
            print(f"Failed for {conversation_id}: {response.status_code}")

        time.sleep(1)  # rate limit safety

    # ------------------------------------
    # SAVE OUTPUT
    # ------------------------------------
    if all_replies_output:

        final_output = {
            "politician": username,
            "posts": all_replies_output
        }

        with open(output_filename, "w", encoding="utf-8") as f:
            json.dump(final_output, f, indent=4)

        print(f"Saved: {output_filename}")
    else:
        print(f"No replies found for {username}")

print("\nAll politicians processed.")


Processing politician: RepWexton
Fetching replies for RepWexton - 1479151898661953543
Fetching replies for RepWexton - 1484643995011661825
Fetching replies for RepWexton - 1487554754976825348
Fetching replies for RepWexton - 1499444166085984259
Fetching replies for RepWexton - 1500955342859030535
Fetching replies for RepWexton - 1506686389148176395
Fetching replies for RepWexton - 1509233047257985034
Fetching replies for RepWexton - 1509654794306265090
Fetching replies for RepWexton - 1511431850903883787
Fetching replies for RepWexton - 1524409962318946304
Fetching replies for RepWexton - 1529159267898499076
Fetching replies for RepWexton - 1554585009570340864
Fetching replies for RepWexton - 1558117201885945856
Fetching replies for RepWexton - 1560679337833025537
Fetching replies for RepWexton - 1567557199660040194
Fetching replies for RepWexton - 1574824888807952391
Fetching replies for RepWexton - 1580269879793315842
Fetching replies for RepWexton - 1587535844361093120
Fetching rep

In [23]:
politicians = """
AlLawsonJr
AlbioSiresNJ
Andy_Levin
Axne4Congress
BHigginsBflo
BennieGThompson
BettyMcCollum04
BillPoseyFL
BobbyLRush
BobbyScott
BradSherman
Call_Me_Dutch
CarlForAlabama
ColinAllredTX
CongBoyle
CongMikeSimpson
CongressmanKean
CongressmanRaja
CongresswomanSC
ConorLambPA
DPrice4Congress
DesJarlaisTN04
DorisMatsui
DrNealDunnFL2
ElaineLuriaVA
EleanorNorton
FrankPallone
GKButterfield
GOPMajorityWhip
GerryConnolly
Hill4MNHouse
Ilhan
JacksonLeeTX18
JakeLaTurner
JayObernolte
JimLangevin
Jim_Jordan
JoaquinCastrotx
JodyHiceFRCA
JohnMannionNY22
JohnYarmuth
JudgeCarter
KellyMorrisonMN
KenCalvert
Kilili_Sablan
MarioDB
MattForMontana
MichelleSteelCA
MikeGarciaCA27
NydiaVelazquez
OAGMaryland
PatronisFL
PeterMeijer
RepAOC
RepAdamGrayCA
RepAdamSmith
RepAdrianSmith
RepAlexMooney
RepAndyBiggsAZ
RepAndyHarrisMD
RepAnnWagner
RepAnnieKuster
RepArrington
RepAshleyHinson
RepBalderson
RepBarbaraLee
RepBarragan
RepBarryMoore
RepBeatty
RepBeccaB
RepBenCline
RepBentz
RepBera
RepBice
RepBillFoster
RepBlakeMoore
RepBobGood
RepBoebert
RepBonnie
RepBost
RepBrandonGill
RepBrecheen
RepBresnahan
RepBrianFitz
RepBrianJack
RepBrianMast
RepBryanSteil
RepBynum
RepCarbajal
RepCarolMiller
RepCartwright
RepCasar
RepCheri
RepClayHiggins
RepCraigGoldman
RepDLamborn
RepDWStweets
RepDanKildee
RepDannyDavis
RepDarrenSoto
RepDaveTaylor
RepDavid
RepDavidKustoff
RepDavidRouzer
RepDavidValadao
RepDavids
RepDeSaulnier
RepDebDingell
RepDeborahRoss
RepDeluzio
RepDerekSchmidt
RepDexterOR
RepDianaDeGette
RepDonBeyer
RepDonaldPayne
RepDrewFerguson
RepDustyJohnson
RepDwightEvans
RepEBJ
RepEllzey
RepEricBurlison
RepEricSorensen
RepEzell
RepFedorchak
RepFeenstra
RepFine
RepFischbach
RepFitzgerald
RepFletcher
RepFrankLucas
RepFredKeller
RepFredUpton
RepFriedmanCA
RepGLatimer
RepGabeAmo
RepGabeVasquez
RepGarbarino
RepGarretGraves
RepGilCisneros
RepGlennIvey
RepGolden
RepGonzalez
RepGosar
RepGregLandsman
RepGregMurphy
RepGregStanton
RepGregSteube
RepGregoryMeeks
RepGrothman
RepGwenMoore
RepHageman
RepHaleyStevens
RepHarshbarger
RepHartzler
RepHorsford
RepHouchin
RepHuizenga
RepJacobs
RepJasonSmith
RepJeffDuncan
RepJeffries
RepJerryNadler
RepJimBaird
RepJimmyPanetta
RepJoeNeguse
RepJohnLarson
RepJohnMcGuire
RepJoshHarder
RepJudyChu
RepJulieJohnson
RepKManning
RepKahele
RepKamlagerDove
RepKatiePorter
RepKenBuck
RepKevinMullin
RepKiggans
RepKiley
RepKnott
RepLSimon
RepLaHood
RepLaLota
RepLaMalfa
RepLaMonica
RepLangworthy
RepLarryBucshon
RepLauraGillen
RepLawrence
RepLiccardo
RepLindaSanchez
RepLisaMcClain
RepLloydDoggett
RepLoriTrahan
RepLouCorrea
RepLoudermilk
RepLuna
RepLuttrell
RepMGS
RepMGriffith
RepMMM
RepMackenzie
RepMalliotakis
RepMaloney
RepMarcyKaptur
RepMariaSalazar
RepMarkAlford
RepMarkHarrisNC
RepMarkTakano
RepMaxMiller
RepMaxwellFrost
RepMcClellan
RepMcDowell
RepMcGarvey
RepMcGovern
RepMcKinley
RepMessmer
RepMeuser
RepMichaelCloud
RepMichaelGuest
RepMichaelRulli
RepMikeCarey
RepMikeCollins
RepMikeKennedy
RepMikeLawler
RepMikeLevin
RepMikeRogersAL
RepMikeTurner
RepMillsPress
RepMoBrooks
RepMoolenaar
RepMoulton
RepMoylan
RepMrvan
RepNancyMace
RepNateMoran
RepNellie
RepNewhouse
RepNickBegich
RepNikkiB
RepOHalleran
RepOgles
RepPabloJose
RepPatHarrigan
RepPatRyanNY
RepPaulTonko
RepPeteStauber
RepPeterDeFazio
RepPressley
RepRalphNorman
RepRaskin
RepRaulRuizMD
RepRichardNeal
RepRickAllen
RepRickLarsen
RepRitchie
RepRoKhanna
RepRobertGarcia
RepRonKind
RepRonnyJackson
RepRoybalAllard
RepRudyYakym
RepRussFulcher
RepRussellFry
RepRutherfordFL
RepRyanZinke
RepSalinas
RepSamGraves
RepSaraJacobs
RepSarahElfreth
RepSchakowsky
RepSchneider
RepScholten
RepSchrader
RepScottPerry
RepScottPeters
RepSeanMaloney
RepSempolinski
RepShontelBrown
RepShriThanedar
RepSpeier
RepStefanik
RepSteveChabot
RepStutzman
RepSuhas
RepSummerLee
RepSusanWild
RepSusieLee
RepSwalwell
RepTenney
RepThomasMassie
RepTimBurchett
RepTimKennedy
RepTimMooreNC
RepTimmons
RepTomBarrett
RepTomSuozzi
RepTonyGonzales
RepTonyWied
RepTroyCarter
RepTroyDowning
RepTroyNehls
RepUnderwood
RepVanTaylor
RepVinceFong
RepVindman
RepWalberg
RepWesleyHunt
RepWilson
RepYoungKim
RepZoeLofgren
Rep_Grijalva
Rep_McBride
Rep_Stansbury
Rep_Walkinshaw
Rep_Whitesides
RobWittman
Robert_Aderholt
RodneyDavis
SanfordBishop
SenMcNerney
SpeakerJohnson
SpeakerMcCarthy
SpeakerPelosi
StaceyPlaskett
SteveScalise
TXRandy14
TimRyan
USRepGaryPalmer
USRepMikeDoyle
USRepMikeFlood
ValerieFoushee
VernBuchanan
WhipClyburn
WhipKClark
auctnr1
boblatta
chelliepingree
davidjtrone
gracenapolitano
jimcoopertn
mattgaetz
michaelcburgess
rep_stevewomack
repblumenauer
repcleaver
repdarrellissa
repdavidscott
repdeliaramirez
repfinstad
repgabeevans
repkevinhern
repmarkpocan
repvalhoyle
rosadelauro
virginiafoxx
"""

politician_list = [p.strip() for p in politicians.split("\n") if p.strip()]

new_df = pd.DataFrame({
    "X Handle": politician_list
})

In [ ]:
AUTOMATE HEALTH-RELATED TWEETS FOR SECOND HALF 

In [29]:
import pandas as pd
import requests
import time
import json

API_BASE_URL = "https://api.data365.co"
API_VERSION = "/v1.1"
API_AUTHORIZATION_TOKEN = "ZXlKMGVYQWlPaUpLVjFRaUxDSmhiR2NpT2lKSVV6STFOaUo5LmV5SnpkV0lpT2lKSWFXSmhRV3hwSWl3aWFXRjBJam94TnpZNU5UQTNOamcyTGpBd05UQXhPSDAuNmpTLTZtbVBGV0F1ZS1McHhXaUIwbUtDUWFDemU4VkRNZVhOSkZ2ckJRTQ=="

# Keyword blocks
KEYWORD_BLOCKS = {
    "1A": "health OR medicine OR medical OR medication",
    "1B": "treatment OR symptoms OR diagnosis",
    "2A": "healthcare OR insurance OR medicare OR medicaid",
    "2B": "premiums OR coverage OR patients OR hospitals",
    "3": "flu OR pneumonia OR bronchitis OR strep throat OR sore throat OR cough OR fever OR infection OR contagious",
    "4": "high blood pressure OR hypertension OR heart OR heart disease OR heart rate OR stroke OR cholesterol OR diabetes OR cancer",
    "5": "lupus OR autoimmune OR chronic illness OR inflammation OR immune system",
    "6": "hydration OR diet OR nutrition OR exercise OR workout OR sleep OR wellness"
}

# ------------------------
# Function to fetch posts only
# ------------------------
def fetch_posts(username, block_name, keywords, max_attempts=10, sleep_time=2):
    update_url = f"{API_BASE_URL}{API_VERSION}/twitter/search/post/update"
    posts_url = f"{API_BASE_URL}{API_VERSION}/twitter/search/post/posts"

    params = {
        "from_profile": username,
        "from_date": "2022-01-01",
        "to_date": "2025-12-31",
        "lang": "en",
        "max_posts": 20,
        "search_type": "latest",
        "max_replies": 3,
        "load_replies": "true",
        "keywords": keywords,
        "access_token": API_AUTHORIZATION_TOKEN
    }

    # Start the search task
    post_response = requests.post(update_url, params=params)
    print(f"Task started for {username} Block {block_name}: {post_response.status_code}")

    attempt = 0
    data = {}
    while attempt < max_attempts:
        posts_response = requests.get(posts_url, params=params)
        if posts_response.status_code == 200:
            data = posts_response.json()
            items = data.get("data", {}).get("items", [])
            if items:
                print(f"{username} Block {block_name}: Retrieved {len(items)} posts")
                break
            else:
                print(f"{username} Block {block_name}, attempt {attempt+1}: No posts yet...")
        else:
            print(f"Failed to fetch posts for {username} (attempt {attempt+1}): {posts_response.status_code}")
        attempt += 1
        time.sleep(sleep_time)

    items = data.get("data", {}).get("items", [])
    if items:
        filename = f"{username} Block {block_name}.json"
        with open(filename, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=4)
        print(f"Saved posts to {filename}")
    else:
        print(f"No posts found for {username} Block {block_name}, skipping save.")

    return items

# ------------------------
# Main Execution
# ------------------------
# Assuming final_df is your DataFrame with a 'X Handle' column
start_username = "RepCasar"
start_index = new_df.index[new_df['X Handle'] == start_username][0]

for username in new_df['X Handle'].iloc[start_index:]:
    print(f"\n=== Processing {username} ===")
    for block_name, keywords in KEYWORD_BLOCKS.items():
        fetch_posts(username, block_name, keywords)


=== Processing RepCasar ===
Task started for RepCasar Block 1A: 202
RepCasar Block 1A: Retrieved 20 posts
Saved posts to RepCasar Block 1A.json
Task started for RepCasar Block 1B: 202
RepCasar Block 1B, attempt 1: No posts yet...
RepCasar Block 1B, attempt 2: No posts yet...
RepCasar Block 1B, attempt 3: No posts yet...
RepCasar Block 1B, attempt 4: No posts yet...
RepCasar Block 1B, attempt 5: No posts yet...
RepCasar Block 1B, attempt 6: No posts yet...
RepCasar Block 1B, attempt 7: No posts yet...
RepCasar Block 1B: Retrieved 14 posts
Saved posts to RepCasar Block 1B.json
Task started for RepCasar Block 2A: 202
RepCasar Block 2A, attempt 1: No posts yet...
RepCasar Block 2A, attempt 2: No posts yet...
RepCasar Block 2A, attempt 3: No posts yet...
RepCasar Block 2A, attempt 4: No posts yet...
RepCasar Block 2A: Retrieved 20 posts
Saved posts to RepCasar Block 2A.json
Task started for RepCasar Block 2B: 202
RepCasar Block 2B, attempt 1: No posts yet...
RepCasar Block 2B, attempt 2: N

In [ ]:
AUTOMATING HEALTH-RELATED REPLIES FOR SECOND HALF

In [35]:
import os
import json
import requests
import time

# ------------------------------------
# CONFIG
# ------------------------------------
BLOCKS = ["1A", "1B", "2A", "2B", "3", "4", "5", "6"]  # correct block names

# ------------------------------------
# START AT AlLawsonJr AND GO DOWNWARD
# ------------------------------------
handles_series = new_df["X Handle"].dropna()

if "RepWalberg" in handles_series.values:
    start_index = handles_series[handles_series == "RepWalberg"].index[0]
    usernames_to_process = new_df.loc[start_index:, "X Handle"].dropna().unique()
else:
    print("RepWalberg not found. Starting from top.")
    usernames_to_process = handles_series.unique()

# ------------------------------------
# LOOP THROUGH EACH POLITICIAN
# ------------------------------------
for username in usernames_to_process:

    print(f"\nProcessing politician: {username}")

    # ------------------------------------
    # LOOP THROUGH BLOCKS
    # ------------------------------------
    for block_name in BLOCKS:

        input_filename = f"{username} Block {block_name}.json"
        output_filename = f"{username} Block {block_name} Replies.json"

        # ------------------------------------
        # SKIP IF FILE DOESN'T EXIST
        # ------------------------------------
        if not os.path.exists(input_filename):
            print(f"Skipping {input_filename} (file not found)")
            continue

        # ------------------------------------
        # OPEN BLOCK FILE
        # ------------------------------------
        with open(input_filename, "r", encoding="utf-8") as f:
            block_data = json.load(f)

        posts = block_data.get("data", {}).get("items", [])

        # Skip if no posts
        if not posts:
            print(f"Skipping {input_filename} (no posts found)")
            continue

        all_replies_output = []

        # ------------------------------------
        # FETCH REPLIES FOR EACH POST
        # ------------------------------------
        for post in posts:
            conversation_id = post.get("conversation_id")

            if not conversation_id:
                continue

            print(f"Fetching replies for {username} Block {block_name} - {conversation_id}")

            replies_url = (
                f"{API_BASE_URL}{API_VERSION}/twitter/profile/"
                f"{username}/feed/posts/{conversation_id}/replies"
            )

            params = {
                "order_by": "date_desc",
                "max_page_size": 50,
                "access_token": API_AUTHORIZATION_TOKEN
            }

            response = requests.get(replies_url, params=params)

            if response.status_code == 200:
                replies_json = response.json()
                replies_items = replies_json.get("data", {}).get("items", [])

                all_replies_output.append({
                    "conversation_id": conversation_id,
                    "replies": replies_items
                })
            else:
                print(f"Failed for {conversation_id}: {response.status_code}")

            time.sleep(1)  # rate limit safety

        # ------------------------------------
        # SAVE OUTPUT ONLY IF REPLIES EXIST
        # ------------------------------------
        if all_replies_output:

            final_output = {
                "politician": username,
                "block": block_name,
                "posts": all_replies_output
            }

            with open(output_filename, "w", encoding="utf-8") as f:
                json.dump(final_output, f, indent=4)

            print(f"Saved: {output_filename}")
        else:
            print(f"No replies found for {username} Block {block_name}")

print("\nAll politicians processed.")


Processing politician: RepWalberg
Fetching replies for RepWalberg Block 1A - 1801340623963779134
Fetching replies for RepWalberg Block 1A - 1807801464544494068
Fetching replies for RepWalberg Block 1A - 1811856379516276792
Fetching replies for RepWalberg Block 1A - 1897761856447451278
Fetching replies for RepWalberg Block 1A - 1907154293779484686
Fetching replies for RepWalberg Block 1A - 1907461308078764475
Fetching replies for RepWalberg Block 1A - 1920851620784373850
Fetching replies for RepWalberg Block 1A - 1926274305689551198
Fetching replies for RepWalberg Block 1A - 1937934872443564186
Fetching replies for RepWalberg Block 1A - 1937962901140062531
Fetching replies for RepWalberg Block 1A - 1938598465258324160
Fetching replies for RepWalberg Block 1A - 1944778177001189791
Fetching replies for RepWalberg Block 1A - 1950903233850880236
Fetching replies for RepWalberg Block 1A - 1961085362983620658
Fetching replies for RepWalberg Block 1A - 1973486977220813168
Fetching replies for

In [ ]:
NON-HEALTH TWEETS FOR SECOND HALF: 

In [37]:
import pandas as pd
import requests
import time
import json

API_BASE_URL = "https://api.data365.co"
API_VERSION = "/v1.1"
API_AUTHORIZATION_TOKEN = "ZXlKMGVYQWlPaUpLVjFRaUxDSmhiR2NpT2lKSVV6STFOaUo5LmV5SnpkV0lpT2lKSWFXSmhRV3hwSWl3aWFXRjBJam94TnpZNU5UQTNOamcyTGpBd05UQXhPSDAuNmpTLTZtbVBGV0F1ZS1McHhXaUIwbUtDUWFDemU4VkRNZVhOSkZ2ckJRTQ=="

# ------------------------
# Function to fetch top posts (no keywords)
# ------------------------
def fetch_top_posts(username, max_attempts=10, sleep_time=2):

    update_url = f"{API_BASE_URL}{API_VERSION}/twitter/search/post/update"
    posts_url = f"{API_BASE_URL}{API_VERSION}/twitter/search/post/posts"

    params = {
        "from_profile": username,
        "from_date": "2022-01-01",
        "to_date": "2025-12-31",
        "lang": "en",
        "max_posts": 20,
        "search_type": "top",   # change to "top" if API supports it
        "max_replies": 3,
        "load_replies": "true",
        "access_token": API_AUTHORIZATION_TOKEN
    }

    # Start search task
    post_response = requests.post(update_url, params=params)
    print(f"Task started for {username}: {post_response.status_code}")

    attempt = 0
    data = {}

    while attempt < max_attempts:
        posts_response = requests.get(posts_url, params=params)

        if posts_response.status_code == 200:
            data = posts_response.json()
            items = data.get("data", {}).get("items", [])

            if items:
                print(f"{username}: Retrieved {len(items)} posts")
                break
            else:
                print(f"{username}, attempt {attempt+1}: No posts yet...")
        else:
            print(f"Failed for {username} (attempt {attempt+1}): {posts_response.status_code}")

        attempt += 1
        time.sleep(sleep_time)

    items = data.get("data", {}).get("items", [])

    if items:
        filename = f"{username}.json"
        with open(filename, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=4)
        print(f"Saved posts to {filename}")
    else:
        print(f"No posts found for {username}, skipping save.")

    return items


# ------------------------
# MAIN EXECUTION
# ------------------------

start_username = "AlLawsonJr"

# Find starting index
start_index = new_df.index[new_df["X Handle"] == start_username][0]

# Loop downward from RepWexton
for username in new_df["X Handle"].iloc[start_index:]:
    print(f"\n=== Processing {username} ===")
    fetch_top_posts(username)

print("\nAll politicians processed.")


=== Processing AlLawsonJr ===
Task started for AlLawsonJr: 202
AlLawsonJr: Retrieved 12 posts
Saved posts to AlLawsonJr.json

=== Processing AlbioSiresNJ ===
Task started for AlbioSiresNJ: 202
AlbioSiresNJ, attempt 1: No posts yet...
AlbioSiresNJ, attempt 2: No posts yet...
AlbioSiresNJ, attempt 3: No posts yet...
AlbioSiresNJ, attempt 4: No posts yet...
AlbioSiresNJ, attempt 5: No posts yet...
AlbioSiresNJ, attempt 6: No posts yet...
AlbioSiresNJ, attempt 7: No posts yet...
AlbioSiresNJ, attempt 8: No posts yet...
AlbioSiresNJ, attempt 9: No posts yet...
AlbioSiresNJ, attempt 10: No posts yet...
No posts found for AlbioSiresNJ, skipping save.

=== Processing Andy_Levin ===
Task started for Andy_Levin: 202
Andy_Levin: Retrieved 25 posts
Saved posts to Andy_Levin.json

=== Processing Axne4Congress ===
Task started for Axne4Congress: 202
Axne4Congress: Retrieved 25 posts
Saved posts to Axne4Congress.json

=== Processing BHigginsBflo ===
Task started for BHigginsBflo: 202
BHigginsBflo: R

In [ ]:
NON-HEALTH TWEETS FOR SECOND HALF: 

In [39]:
import os
import json
import requests
import time

# ------------------------------------
# START AT RepWexton AND GO DOWNWARD
# ------------------------------------
handles_series = new_df["X Handle"].dropna()

if "AlLawsonJr" in handles_series.values:
    start_index = handles_series[handles_series == "AlLawsonJr"].index[0]
    usernames_to_process = new_df.loc[start_index:, "X Handle"].dropna().unique()
else:
    print("AlLawsonJr not found. Starting from top.")
    usernames_to_process = handles_series.unique()

# ------------------------------------
# LOOP THROUGH EACH POLITICIAN
# ------------------------------------
for username in usernames_to_process:

    print(f"\nProcessing politician: {username}")

    input_filename = f"{username}.json"
    output_filename = f"{username} Replies.json"

    # ------------------------------------
    # SKIP IF POSTS FILE DOESN'T EXIST
    # ------------------------------------
    if not os.path.exists(input_filename):
        print(f"Skipping {input_filename} (file not found)")
        continue

    # ------------------------------------
    # OPEN POSTS FILE
    # ------------------------------------
    with open(input_filename, "r", encoding="utf-8") as f:
        post_data = json.load(f)

    posts = post_data.get("data", {}).get("items", [])

    if not posts:
        print(f"No posts found in {input_filename}, skipping.")
        continue

    all_replies_output = []

    # ------------------------------------
    # FETCH REPLIES FOR EACH POST
    # ------------------------------------
    for post in posts:
        conversation_id = post.get("conversation_id")

        if not conversation_id:
            continue

        print(f"Fetching replies for {username} - {conversation_id}")

        replies_url = (
            f"{API_BASE_URL}{API_VERSION}/twitter/profile/"
            f"{username}/feed/posts/{conversation_id}/replies"
        )

        params = {
            "order_by": "date_desc",
            "max_page_size": 50,
            "access_token": API_AUTHORIZATION_TOKEN
        }

        response = requests.get(replies_url, params=params)

        if response.status_code == 200:
            replies_json = response.json()
            replies_items = replies_json.get("data", {}).get("items", [])

            all_replies_output.append({
                "conversation_id": conversation_id,
                "replies": replies_items
            })
        else:
            print(f"Failed for {conversation_id}: {response.status_code}")

        time.sleep(1)  # rate limit safety

    # ------------------------------------
    # SAVE OUTPUT
    # ------------------------------------
    if all_replies_output:

        final_output = {
            "politician": username,
            "posts": all_replies_output
        }

        with open(output_filename, "w", encoding="utf-8") as f:
            json.dump(final_output, f, indent=4)

        print(f"Saved: {output_filename}")
    else:
        print(f"No replies found for {username}")

print("\nAll politicians processed.")


Processing politician: AlLawsonJr
Fetching replies for AlLawsonJr - 1553127999846989826
Fetching replies for AlLawsonJr - 1553473426085527554
Fetching replies for AlLawsonJr - 1554565066263089153
Fetching replies for AlLawsonJr - 1554565214229745664
Fetching replies for AlLawsonJr - 1554626594903396352
Fetching replies for AlLawsonJr - 1555186403360768001
Fetching replies for AlLawsonJr - 1559534224981491712
Fetching replies for AlLawsonJr - 1562036343005380608
Fetching replies for AlLawsonJr - 1570031195911749632
Fetching replies for AlLawsonJr - 1578527816831811584
Fetching replies for AlLawsonJr - 1580180550161047552
Fetching replies for AlLawsonJr - 1580181288723525633
Saved: AlLawsonJr Replies.json

Processing politician: AlbioSiresNJ
Skipping AlbioSiresNJ.json (file not found)

Processing politician: Andy_Levin
Fetching replies for Andy_Levin - 1484901000435707908
Fetching replies for Andy_Levin - 1493618115355320332
Fetching replies for Andy_Levin - 1496506383642079236
Fetching

In [2]:
# 1. Imports
import os
import json
import pandas as pd

# 2. Storage container
master_data = {}

# 3. Loop through folder JSONs
for filename in os.listdir():
    if not filename.endswith(".json"):
        continue

    with open(filename, "r", encoding="utf-8") as f:
        data = json.load(f)

    # ---------- HEALTH TWEETS ----------
    if "Block" in filename and "Replies" not in filename:

        username = filename.split(" Block ")[0]
        block = filename.split("Block ")[1].replace(".json", "")

        posts = data.get("data", {}).get("items", [])

        for post in posts:

            conversation_id = post.get("conversation_id")

            key = (username, conversation_id)

            master_data[key] = {
                "username": username,
                "block": block,
                "tweet_type": "health",
                "tweet_text": post.get("text"),
                "created_time": post.get("created_time"),
                "favorite_count": post.get("favorite_count"),
                "reply_count": post.get("reply_count"),
                "retweet_count": post.get("retweet_count"),
                "quote_count": post.get("quote_count"),
                "view_count": post.get("view_count"),
                "replies": []
            }

    # ---------- HEALTH REPLIES ----------
    elif "Block" in filename and "Replies" in filename:

        username = filename.split(" Block ")[0]
        block = filename.split("Block ")[1].replace(" Replies.json", "")

        posts = data.get("posts", [])

        for post in posts:

            conversation_id = post.get("conversation_id")
            replies = post.get("replies", [])

            key = (username, conversation_id)

            if key in master_data:
                for reply in replies:
                    master_data[key]["replies"].append(reply.get("text"))

    # ---------- NON-HEALTH TWEETS ----------
    elif "Block" not in filename and "Replies" not in filename:

        username = filename.replace(".json", "")

        posts = data.get("data", {}).get("items", [])

        for post in posts:

            conversation_id = post.get("conversation_id")

            key = (username, conversation_id)

            master_data[key] = {
                "username": username,
                "block": None,
                "tweet_type": "non_health",
                "tweet_text": post.get("text"),
                "created_time": post.get("created_time"),
                "favorite_count": post.get("favorite_count"),
                "reply_count": post.get("reply_count"),
                "retweet_count": post.get("retweet_count"),
                "quote_count": post.get("quote_count"),
                "view_count": post.get("view_count"),
                "replies": []
            }

    # ---------- NON-HEALTH REPLIES ----------
    elif "Replies" in filename and "Block" not in filename:

        username = filename.replace(" Replies.json", "")

        posts = data.get("posts", [])

        for post in posts:

            conversation_id = post.get("conversation_id")
            replies = post.get("replies", [])

            key = (username, conversation_id)

            if key in master_data:
                for reply in replies:
                    master_data[key]["replies"].append(reply.get("text"))

# 4. Convert dictionary → dataframe (ONE ROW PER TWEET OR REPLY)

rows = []

for key, value in master_data.items():

    # ----- Add the original tweet row -----
    rows.append({
        "username": value["username"],
        "block": value["block"],
        "tweet_type": value["tweet_type"],
        "conversation_id": key[1],
        "text": value["tweet_text"],
        "row_type": "tweet",
        "created_time": value["created_time"],
        "favorite_count": value["favorite_count"],
        "reply_count": value["reply_count"],
        "retweet_count": value["retweet_count"],
        "quote_count": value["quote_count"],
        "view_count": value["view_count"]
    })

    # ----- Add one row per reply -----
    for reply in value["replies"]:
        if reply and reply.strip() != "":
            rows.append({
                "username": value["username"],
                "block": value["block"],
                "tweet_type": value["tweet_type"],
                "conversation_id": key[1],
                "text": reply,
                "row_type": "reply",
                "created_time": None,
                "favorite_count": None,
                "reply_count": None,
                "retweet_count": None,
                "quote_count": None,
                "view_count": None
            })

final_df = pd.DataFrame(rows)

# 5. Save output
final_df.to_csv("ALL_TWEETS_MASTER.csv", index=False)

print("Master dataset created!")

Master dataset created!


In [17]:
all_tweets = pd.read_csv("ALL_TWEETS_MASTER.csv")
all_tweets.head()

,username,block,tweet_type,conversation_id,text,row_type,created_time,favorite_count,reply_count,retweet_count,quote_count,view_count
0,RepGwenMoore,2B,health,1940143080637964748,What did the American people buy with the $4 t...,tweet,2025-07-01T20:18:51,22.0,12.0,14.0,2.0,1636.0
1,RepGwenMoore,2B,health,1940143080637964748,@RepGwenMoore https://t.co/g3hDlL5JSW,reply,NaN,NaN,NaN,NaN,NaN,NaN
2,RepGwenMoore,2B,health,1940143080637964748,@RepGwenMoore Yawn.,reply,NaN,NaN,NaN,NaN,NaN,NaN
3,RepGwenMoore,2B,health,1940143080637964748,THANK YOU for pointing out that removing the u...,reply,NaN,NaN,NaN,NaN,NaN,NaN
4,RepGwenMoore,2B,health,1940143080637964748,@RepGwenMoore if your a dorky lookin guy you ...,reply,NaN,NaN,NaN,NaN,NaN,NaN


In [19]:
all_tweets["username"].nunique()

#out of the 713 initial politicians, only 658 ended up included. This means that 55 politicians had no tweets that matched the health keywords

658

In [21]:
all_tweets = all_tweets.drop_duplicates(subset=["conversation_id", "text"])

In [9]:
len(all_tweets)

207956

In [16]:
all_tweets.to_csv("ALL_TWEETS_MASTER_FINAL.csv", index=False)

In [61]:
# takes a 2 column csv, adds a third column
#first column is ID, second column is the text 
#label studio then allows you to give it a labeling structure (binary structure)
#once you get ~10-15 positives and negatives and starts running a model and gives the ones that are hard to classify, would have to classify myself
# theres an option to do a test, of the performance of the model 
#should't be a final model, should do separate model on python but still useful to tell you which rows are most useful and how well the class. model is working
#multiclass labeling where also labeling sentiment along with the misinformation. before that, need to know if the replies are replying to the misinformation or not. do classification model first for main tweets, and once that performs well, take that classification, put it in as part of the text 
# 3 dolumns : conversatin id, text, 0 or 1 to label misinformation
#replies i want to look at sentiment
#were they part of a controversy, covid, etc are things i may also end up labeling for
# need at about over 80% accuracy 

In [23]:
tweets_df = all_tweets[
    (all_tweets["row_type"] == "tweet") & (all_tweets["tweet_type"] == "health")
][["conversation_id", "text"]].copy()

tweets_label = tweets_df.copy()

# Add ID column
tweets_label["id"] = range(1, len(tweets_label) + 1)

# Keep only needed columns
tweets_label = tweets_label[["id", "conversation_id", "text"]]

# Save to CSV
tweets_label.to_csv("tweet_labeling.csv", index=False)

In [25]:
#create replies only df:

replies_df = all_tweets[
    (all_tweets["row_type"] == "reply") & (all_tweets["tweet_type"] == "health")
][["conversation_id", "text"]].copy()

replies_label = replies_df.copy()

replies_label["id"] = range(1, len(replies_label) + 1)

replies_label = replies_label[["id", "conversation_id", "text"]]

replies_label.to_csv("reply_labeling.csv", index=False)